In [ ]:
라이브러리 가져오기

In [31]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import PyPDFLoader, CSVLoader
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

load_dotenv()   # .env 파일에 저장되어있는 api 키 가져오기

True

모델설정 및 DB 설정

In [ ]:
# 모델 설정
model = init_chat_model("openai:gpt-5.6-luna")

# 벡터 저장소 설정
# 임베딩 및 저장

# 경로
DB_PATH = "../data/chroma_3store"
csv_path = "../data/16-6_개인정보FAQ.csv"
pdf_paths = [
    "../data/16-1_K희망사다리2026_모두의정책.pdf",
    "../data/Samsung_Electronics_Sustainability_Report_2026_KOR.pdf",
]

# CSV 로드
csv_docs = CSVLoader( # csv는 dict와 다르게 encoding 인자를 받지 않음
    file_path=csv_path,
    encoding="utf-8-sig", #cp949는 csv_read이고 현재는 
).load()


# PDF 로드
pdf_docs = []

for pdf_path in pdf_paths:
    pdf_docs.extend(
        PyPDFLoader(pdf_path).load()
    )

# 전체 문서 합치기
documents = pdf_docs + csv_docs

# 임베딩 모델 설정
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 질문과 가까운 부분만 찾을 가능성 높이기 위해 청크 사용 -  관련 내용 조각화
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
)
split_documents = text_splitter.split_documents(documents)

# 벡터 검색은 문서전체 X, 질문과 가장 비슷하고 작은
# 검색결과 너무 많은 내용을 포함 -> 청크 작게
# 답변 시 필요 문맥이 끊긴다 -> 청크 크게

In [ ]:
# PDF 1 
pdf1_docs = PyPDFLoader(
    str(pdf_paths[0])
).load()

for doc in pdf1_docs:
    doc.metadata.update({
        "source_type": "pdf",
        "source_file": Path(pdf_paths[0]).name, #Path 라이브러리 필요
        "document_id": "policy_hope_ladder",
    })



In [ ]:
#   print(pdf_paths[0]) # 해당 컬럼 존재하는 지 확인

../data/16-1_K희망사다리2026_모두의정책.pdf


In [33]:
# PDF 2
pdf2_docs = PyPDFLoader(
    str(pdf_paths[1])
).load()

for doc in pdf2_docs:
    doc.metadata.update({
        "source_type": "pdf",
        "source_file": Path(pdf_paths[1]).name,
        "document_id": "samsung_sustainability",
    })

In [34]:
# csv
for row_number, doc in enumerate(csv_docs): #
    doc.metadata.update({
        "source_type": "csv",
        "source_file": Path(csv_path).name,
        "document_id": "privacy_faq",
        "row": row_number,
    })

In [35]:
#벡터 DB 생성 및 저장
vectorstore = Chroma.from_documents(
    documents= split_documents,  #- split_documents: PDF와 CSV에서 만든 분할 문서
    embedding=embeddings, #
    collection_name="3store", #
    persist_directory=str(DB_PATH), # 
)

#저장된 벡터 DB 불러오기
load_vs = Chroma(
    collection_name="3store", #
    embedding_function=embeddings, #
    persist_directory=str(DB_PATH), #
)

# # 저장된 벡터 DB 가져와야
# load_vs = Chroma(
#     collection_name="3store",
#     embedding_function=embeddings,
#     persist_directory=DB_PATH
# )

In [ ]:
documents = pdf1_docs + pdf2_docs + csv_docs
print("전체 문서 수:", len(documents))

전체 문서 수: 268


In [37]:
# print("PDF 문서 수:", len(pdf_docs))
# print("CSV 문서 수:", len(csv_docs))
print(pdf_docs[0].metadata)
print(csv_docs[0])

{'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.4 (Macintosh)', 'creationdate': '2026-02-09T17:43:27+09:00', 'moddate': '2026-03-03T16:13:53+09:00', 'trapped': '/False', 'source': '../data/16-1_K희망사다리2026_모두의정책.pdf', 'total_pages': 268, 'page': 0, 'page_label': '1'}
page_content='처리상황단계내용: 제공
적용분야내용: 금융 분야
개인정보유형내용: 일반정보
코드제목: 채무보증인의 상속인에게 채무자의 개인정보 제공 가능?
주제내용: 채무보증인의 상속인에게 원 채무자의 개인정보 제공 정당성
문제상황내용: A는 직무수행 중 부상을 당해 전역하였으며, 전역 후 사업을 하기위해 국가유공자 신분으로 B를 보증인으로 하여 은행으로부터 유리한 조건으로 대출을 받았습니다. A는 어느 날부터 원리금을 미납하기 시작하여 은행은 전화 및 우편 등을 통해 A에게 연락을 취하였으나 전화번호는 변경되었고, 발송된 우편물은 수신자 미거주로 반송되었습니다. 이후 A의 보증인 B마저 사망하자 은행은 B의 상속인에게 A의 대부원리금 상환을 요구하며 부동산과 예금에 가압류 조치를 할 수 있음을 통보하였습니다. B의 상속인은 대부원리금 상환 요구에 대한 억울함을 호소하며 은행에게 채무자 A의 연락처를 알려달라고 합니다.
질문: 상속인의 요청에 따라 은행은 채무자 A의 연락처(변경 전 전화번호, 우편물 반송된 주소 등을 말함)를 상속인에게 제공할 수 있는지요?
해결방법내용: 개인정보보호법에 따라 개인정보처리자(은행)는 정보주체 또는 제3자의 이익을 부당하게 침해할 우려가 있을 때를 제외하고 정보주체 또는 그 법정대리인이 주소불명 등으로 사전 동의를 받을 수 없는 경우로, 명백히 정보주체 또는 제3자의 급박한 생명, 신체, 재산의 이익을 위하여 

In [ ]:
# 검색기
retreiver_mmr = load_vs.as_retriever(search_type="mmr",
                                     search_kwargs={"k": 5, 
                                                    "fetch_k": 50,
                                                    "lambda_mult" : 0.25})

In [ ]:
# RAG 로 붙여보기
SYSTEM_PROMPT = """
너는 공공 정책 안내 도우미다.
아래 자료를 참고해서 답해라. 
참고 자료에 없으면 "자료에 없음" 이라고 말해라
정확한 자격, 금액, 기한은 공고 확인이 필요하다고 꼭 덧붙여라.
답 끝에 참고한 페이지 번호를 [p.60] 과 같은 형식으로 표시해라.
"""

# 검색 결과 문서를 받았을 때 메타데이터와 내용을 합쳐서 text 로 반환하는 함수 작성
def format_docs(docs):
    context = ""

    for doc in docs:
        context += f"[p.{doc.metadata['page']}] \n {doc.page_content} \n\n"

    return context

In [ ]:
chain = retreiver_mmr | format_docs
chain

In [ ]:
# 확인용
chain.invoke("청년 월세 지원 정책 찾아줘")

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# rag_prompt 완성해보기
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human', "참고자료\n{context} \n질문{question}")
])

                            
rag_chain = (
    {"context" : ( retreiver_mmr | format_docs ), "question" :  RunnablePassthrough()} # question 은 검색기를 거쳐서 문서찾아서 context 키값의 벨류 
    | rag_prompt  # question 은 그대로 prompt 에 들어가야
    | model
    | StrOutputParser()
    )

rag_chain.invoke("청년 월세 지원 정책 찾아줘")

NameError: name 'SYSTEM_PROMPT' is not defined

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field

# 구조화된 출력 받기
from pydantic import BaseModel, Field

class AnswerStyle(BaseModel):
    answer : str = Field(description="최종 답변")
    source : str = Field(description="출처")

structured_model = model.with_structured_output(AnswerStyle, method="json_mode")
                            
rag_chain = (
    {"context" : ( retreiver_mmr | format_docs ), "question" :  RunnablePassthrough()} # question 은 검색기를 거쳐서 문서찾아서 context 키값의 벨류 
    | rag_prompt  # question 은 그대로 prompt 에 들어가야
    | structured_model
    )

result = rag_chain.invoke("청년 월세 지원 정책 찾아줘")

In [ ]:
result.model_dump()

In [ ]:
model.invoke("신대방 삼거리 맛집 알려줘")

In [ ]:
structured_model.invoke("신대방 삼거리 맛집 알려줘").model_dump()

In [ ]:
# 초등학생도 풀 수 있는 RAG 질문 연습
practice_questions = [
    '이 문서는 어떤 정책을 설명하고 있나요?',
    '청년 월세 지원은 누구를 위한 제도인가요?',
    '신청할 때 확인해야 할 것은 무엇인가요?',
    '지원 내용에서 꼭 기억할 숫자는 무엇인가요?',
    '더 자세히 물어보려면 어디에 문의하면 되나요?',
]

for number, question in enumerate(practice_questions, start=1):
    print(f'문제 {number}: {question}')
    answer = rag_chain.invoke(question)
    print(answer)
    print('-' * 60)

## 연습 문제 확인표

각 답변을 보고 다음 세 가지를 확인해 보세요.

1. 질문에 바로 답했나요?
2. 문서에 없는 내용을 상상해서 말하지 않았나요?
3. 답변에 참고 페이지가 표시되었나요?

답변이 이상하면 검색 결과 개수 `k`를 3 또는 5로 바꾸고 다시 실행해 보세요.


In [ ]:
# 내가 만든 질문 하나로 다시 테스트하기
my_question = '청년 월세 지원을 받으려면 무엇을 먼저 확인해야 하나요?'
my_answer = rag_chain.invoke(my_question)
print(my_answer)

## 가장 간단한 RAG 실습

질문을 하면 관련 문서를 찾고, 그 문서를 읽은 AI가 답변합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

simple_prompt = ChatPromptTemplate.from_template("""
아래 문서만 참고해서 질문에 답해줘.
문서에 답이 없으면 '문서에서 찾지 못했어요'라고 말해줘.

문서:
{context}

질문:
{question}
""")

simple_chain = (
    {
        'context': retreiver_mmr | format_docs,
        'question': RunnablePassthrough(),
    }
    | simple_prompt
    | model
    | StrOutputParser()
)

simple_chain.invoke('청년 월세 지원은 누구를 위한 제도인가요?')